# Ejercicio 01 — Mi primer push con Git

---

## Objetivo

Hacer tu **primer push con Git** al repositorio de la materia: crear tu rama personal, registrar tus datos y abrir tu primer Pull Request.

## ¿Qué se verifica?

El script chequea **3 niveles progresivos** de tu setup de Git:

| Nivel | Verifica | Requiere |
| :---: | :--- | :--- |
| 1 | Git instalado | `git` disponible en la terminal |
| 2 | Identidad configurada | `git config user.name` y `user.email` seteados |
| 3 | Repo + rama personal | Estás en tu clon del repo de la materia, parado en `estudiante/<apellido>-<nombre>` (coincidiendo con tus datos del Paso 1) |

> **Sobre la entrega**: leé [`README.md`](README.md) en esta carpeta — el deliverable es **un `.txt` por estudiante** en `estudiantes/`, no el notebook.

## ⚠️ Antes de empezar: verificá tu rama

Para subir tu entrega tenés que estar parado en **tu rama personal** `estudiante/apellido-nombre`. Si estás en `main` o `dev`, vas a romper el patrón de PR del curso.

**Cómo se arma el nombre**: minúsculas, sin tildes ni eñes, y **un solo guión** separando apellido de nombre. Si tenés más de un nombre o más de un apellido, **van pegados**:

| Estudiante | Rama |
| :--- | :--- |
| Juan Sokil | `estudiante/sokil-juan` |
| Juan Pablo Sokil | `estudiante/sokil-juanpablo` |
| María José García López | `estudiante/garcialopez-mariajose` |
| Tomás Del Río | `estudiante/delrio-tomas` |

El guión es el separador apellido-nombre: si escribieras `garcia-lopez-maria-jose` nadie sabría dónde termina el apellido. Por eso va **exactamente uno**, y un robot de GitHub rechaza los PRs de ramas que no cumplen.

```bash
# Ver en qué rama estás
git branch --show-current

# Si todavía no la creaste (primera vez):
git checkout -b estudiante/apellido-nombre   # reemplazá por tu apellido-nombre

# Si ya existe y estás en otra rama:
git checkout estudiante/apellido-nombre
```

> El **Paso 2** de abajo verifica esto programáticamente: calcula tu rama a partir de los datos del Paso 1 y, si no coincide, te da el comando exacto para renombrarla.

---

## Paso 1 — Completá tus datos

Editá la siguiente celda con tu **nombre**, **apellido** y **usuario de GitHub**, y ejecutala:

In [ ]:
nombre = ''         # <-- Completar con tu nombre
apellido = ''       # <-- Completar con tu apellido
usuario_github = '' # <-- Completar con tu usuario de GitHub (sin @)

---

## Paso 2 — Verificá tu setup de Git

Ejecutá la celda de abajo. Verifica los 3 niveles (Git instalado → identidad configurada → rama personal) y al final calcula un **código de verificación** de 12 caracteres que vas a entregar.

In [ ]:
import hashlib, subprocess, unicodedata, re
from datetime import date

if not nombre.strip() or not apellido.strip() or not usuario_github.strip():
    raise ValueError('Completa nombre, apellido y usuario de GitHub en la celda anterior antes de ejecutar.')

resultados = {}
nivel_alcanzado = 0


def git(*args):
    return subprocess.check_output(['git', *args], stderr=subprocess.DEVNULL).decode().strip()


def slug(s):
    """Normaliza un nombre o apellido a una sola palabra.

    Los compuestos van PEGADOS: el guion se reserva como separador
    apellido-nombre, asi la rama nunca es ambigua.
        'Juan Pablo'   -> 'juanpablo'
        'Garcia Lopez' -> 'garcialopez'
        'D'Amato'     -> 'damato'
    """
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^a-zA-Z0-9]+', '', s).lower()   # sin espacios, guiones ni apostrofes
    return s


rama_esperada = f'estudiante/{slug(apellido)}-{slug(nombre)}'

# ============================================================
# NIVEL 1: Git instalado
# ============================================================
print('--- Nivel 1: Git instalado ---')
try:
    version = git('--version')
    print(f'  {version}: OK')
    resultados['nivel1'] = 'OK'
    nivel_alcanzado = 1
except Exception:
    print('  git: NO ENCONTRADO')
    print('  -> Instala Git desde https://git-scm.com/downloads')
    resultados['nivel1'] = 'FALLO'

print()

# ============================================================
# NIVEL 2: Identidad configurada
# ============================================================
print('--- Nivel 2: Identidad configurada ---')
try:
    g_name = git('config', 'user.name')
except Exception:
    g_name = ''
try:
    g_email = git('config', 'user.email')
except Exception:
    g_email = ''

if g_name and g_email:
    print(f'  user.name:  {g_name}')
    print(f'  user.email: {g_email}')
    resultados['nivel2'] = f'OK ({g_name})'
    nivel_alcanzado = 2
else:
    print('  Identidad incompleta.')
    print('  -> Configurala:')
    print("       git config --global user.name 'Tu Nombre'")
    print("       git config --global user.email 'tu@email.com'")
    resultados['nivel2'] = 'FALLO'

print()

# ============================================================
# NIVEL 3: Repo del curso + rama personal correcta
# ============================================================
print('--- Nivel 3: Repo del curso + rama personal ---')

# 3a. ¿Estas parado en el repo de la materia?
try:
    remote_url = git('remote', 'get-url', 'origin')
except Exception:
    remote_url = ''

repo_ok = 'LCD-InfraCienciaDatos' in remote_url

# 3b. ¿Estas en TU rama, con la convencion del curso?
try:
    rama_actual = git('branch', '--show-current')
except Exception:
    rama_actual = '(no detectada)'

if not repo_ok:
    print(f'  Remote origin: {remote_url or "(no detectado)"} — NO es el repo de la materia.')
    print('  -> Abri este notebook desde tu clon de LCD-InfraCienciaDatos')
    print('     (ver README de la clase, Paso 1: git clone).')
    resultados['nivel3'] = 'FALLO (repo equivocado)'
elif rama_actual in ('main', 'master', 'dev', '', '(no detectada)'):
    print(f'  Rama actual: {rama_actual!r} — NO es tu rama personal.')
    print(f'  -> Crea/movete a tu rama: git checkout -b {rama_esperada}')
    resultados['nivel3'] = 'FALLO (rama no personal)'
elif not rama_actual.startswith('estudiante/'):
    print(f'  Rama actual: {rama_actual!r} — le falta el prefijo estudiante/.')
    print(f'  -> La convencion del curso es: {rama_esperada}')
    print(f'     Renombrala con: git branch -m {rama_esperada}')
    resultados['nivel3'] = 'FALLO (sin prefijo estudiante/)'
elif rama_actual != rama_esperada:
    print(f'  Rama actual: {rama_actual!r} — no coincide con tus datos del Paso 1.')
    print(f'  -> Segun tu nombre y apellido, tu rama debe ser: {rama_esperada}')
    print(f'     Renombrala con: git branch -m {rama_esperada}')
    print('     (Si el error esta en el Paso 1, corregilo ahi y re-ejecuta ambas celdas.)')
    resultados['nivel3'] = 'FALLO (rama no coincide con tus datos)'
else:
    print('  Remote origin: OK (repo de la materia)')
    print(f'  Rama actual: {rama_actual!r}: OK')
    resultados['nivel3'] = f'OK ({rama_actual})'
    nivel_alcanzado = 3

print()

# ============================================================
# RESULTADO FINAL
# ============================================================
codigo_raw = f'{apellido.strip().lower()}-{nombre.strip().lower()}-nivel{nivel_alcanzado}-{date.today().isoformat()}'
codigo = hashlib.sha256(codigo_raw.encode()).hexdigest()[:12].upper()

print('=' * 52)
print('       VERIFICACION GIT - EJERCICIO 01')
print('=' * 52)
print(f'Estudiante: {nombre.strip()} {apellido.strip()} (@{usuario_github.strip()})')
print()
print(f"  Nivel 1 (Git instalado):   {resultados.get('nivel1', 'NO EJECUTADO')}")
print(f"  Nivel 2 (Identidad):       {resultados.get('nivel2', 'NO EJECUTADO')}")
print(f"  Nivel 3 (Repo + rama):     {resultados.get('nivel3', 'NO EJECUTADO')}")
print()
print(f'  Nivel alcanzado: {nivel_alcanzado} / 3')
print(f'  Codigo: {codigo}')
print('=' * 52)
print()
if nivel_alcanzado == 3:
    print('Setup de Git completo. Pasa al Paso 3 para generar tu archivo de entrega.')
elif nivel_alcanzado > 0:
    print('Verificacion parcial. Revisa el troubleshooting de abajo.')
    print('Igual podes generar el archivo de entrega (Paso 3) con el nivel alcanzado.')
else:
    print('Ninguna verificacion paso. Revisa el troubleshooting antes de generar el archivo.')

---

## Paso 3 — Generá tu archivo de entrega

La siguiente celda toma tu nombre/apellido, los normaliza (sin tildes, minúsculas, separado por guión) y crea el archivo:

```
clase01/ejercicios/estudiantes/<apellido>-<nombre>.txt
```

Antes de escribir, **te muestra el filename y pide confirmación**. Si está mal, contestá `n`, corregí la celda del Paso 1 y volvé a correr.

> **Por qué este paso existe**: el `.ipynb` es compartido — si todos lo modifican y commitean, los PRs colisionan. La entrega es un archivo único por estudiante en `estudiantes/`. Más detalles en [`README.md`](README.md).

In [ ]:
import subprocess
from pathlib import Path
from datetime import date  # por si esta celda se corre sin haber pasado por el Paso 2

# --- Prerrequisito: el codigo de verificacion del Paso 2 ---
if 'codigo' not in globals():
    raise RuntimeError('Falta el codigo de verificacion: ejecuta primero el Paso 2.')

# --- Ubicar la raiz del repo (funciona sin importar desde donde abriste Jupyter) ---
try:
    repo_root = Path(subprocess.check_output(
        ['git', 'rev-parse', '--show-toplevel'],
        stderr=subprocess.DEVNULL,
    ).decode().strip())
except Exception:
    raise RuntimeError(
        'No estas parado dentro de un repo git. Abri este notebook desde tu clon de la materia.'
    )

# --- Verificar rama actual antes de generar el archivo ---
try:
    rama_actual = subprocess.check_output(
        ['git', 'branch', '--show-current'],
        stderr=subprocess.DEVNULL,
    ).decode().strip()
except Exception:
    rama_actual = '(no detectada)'

if rama_actual in ('main', 'master', 'dev', '', '(no detectada)'):
    print(f'⚠️  ATENCION: estas en la rama {rama_actual!r}.')
    print('    Antes de subir tu entrega tenes que estar en TU rama personal')
    print('    estudiante/apellido-nombre. Desde la terminal:')
    print('        git checkout -b estudiante/apellido-nombre   (reemplaza por tu apellido-nombre)')
    print()
    print('    Igual generamos el archivo, pero acordate del checkout antes del push.')
    print()
else:
    print(f'OK — rama actual: {rama_actual!r}.')
    print()

apellido_slug = slug(apellido)   # slug() quedo definido en el Paso 2
nombre_slug = slug(nombre)

if not apellido_slug or not nombre_slug:
    print('No se pudo generar un nombre de archivo valido. Revisa nombre/apellido en el Paso 1.')
else:
    filename = f'{apellido_slug}-{nombre_slug}.txt'
    target = repo_root / 'clase01' / 'ejercicios' / 'estudiantes' / filename
    target_rel = target.relative_to(repo_root).as_posix()

    _verbo = 'SOBRESCRIBIR (ya existe)' if target.exists() else 'crear'
    print(f'Voy a {_verbo}: {target_rel}')
    try:
        confirm = input('Confirmas? (s/n): ').strip().lower()
    except Exception:
        # Entorno sin stdin interactivo (VS Code remoto, ejecucion automatizada):
        # el filename ya se mostro arriba — si esta mal, corregi el Paso 1 y re-ejecuta.
        print('(entorno sin input interactivo — se confirma automaticamente)')
        confirm = 's'

    if confirm in ('s', 'si', 'y', 'yes'):
        target.parent.mkdir(parents=True, exist_ok=True)
        contenido = (
            f'Apellido: {apellido.strip()}\n'
            f'Nombre: {nombre.strip()}\n'
            f'Usuario GitHub: {usuario_github.strip()}\n'
            f'Rama: {rama_actual}\n'
            f'Codigo: {codigo}\n'
            f'Fecha: {date.today().isoformat()}\n'
        )
        target.write_text(contenido, encoding='utf-8')
        print()
        print(f'Archivo creado: {target_rel}')
        print()
        print('Ahora subi SOLO ese archivo (Paso 4):')
        print(f'  git add {target_rel}')
        print('  git commit -m "ejercicio01: registro"')
        print(f'  git push origin {rama_actual if rama_actual.startswith("estudiante/") else "estudiante/apellido-nombre"}')
    else:
        print('No se escribio nada. Volve a correr esta celda cuando quieras confirmar.')

---

## 🛠️ Troubleshooting

| Problema | Solución |
| :--- | :--- |
| **Nivel 1 falla** (git no encontrado) | Instalá Git desde https://git-scm.com/downloads y reiniciá la terminal/VSCode |
| **Nivel 2 falla** (identidad incompleta) | Configurá tu identidad: `git config --global user.name "Tu Nombre"` y `git config --global user.email "tu@email.com"` |
| **Nivel 3 falla** (repo equivocado) | Estás parado en otra carpeta/repo. Abrí el notebook desde tu clon de `LCD-InfraCienciaDatos` |
| **Nivel 3 falla** (rama no esperada) | El propio mensaje te dice el nombre correcto (`estudiante/<apellido>-<nombre>`) y el comando exacto (`git checkout -b ...` o `git branch -m ...`) |
| **El Paso 3 dice "ejecuta primero el Paso 2"** | Las celdas se corren en orden: Paso 1 → Paso 2 → Paso 3 (el código de verificación se calcula en el Paso 2) |
| **`git push` pide password** | Usá un **Personal Access Token (PAT)** de GitHub como contraseña. Ver el [`README.md` de la clase](../README.md) para generarlo |

---

## Paso 4 — Subí tu entrega

**Solo subí el archivo `.txt`** que se creó en el Paso 3. **No commitees** el `ejercicio.ipynb` modificado: es un template compartido y generaría conflictos con el resto de los estudiantes.

Desde la raíz del repo, reemplazá `<apellido>-<nombre>` por el filename que te imprimió el Paso 3:

```bash
git add clase01/ejercicios/estudiantes/<apellido>-<nombre>.txt
git commit -m "ejercicio01: registro"
git push origin estudiante/apellido-nombre
```

> **Nota**: si es la primera vez que pusheás esta rama, Git puede pedir `git push --set-upstream origin estudiante/apellido-nombre`. Es normal, solo pasa la primera vez.

---

## Paso 5 — Abrí tu Pull Request (una sola vez en todo el curso)

Es tu **primer y único** Pull Request del curso. Lo abrís ahora y lo dejás **abierto** todo el cuatrimestre:

1. En GitHub, **"Compare & pull request"** sobre tu rama `estudiante/apellido-nombre`.
2. Título: dejá el que sugiere GitHub (el nombre de tu rama).
3. Creá el PR y **dejalo abierto**. El docente revisa tus entregas ahí.

### Cómo leer tu PR de acá en adelante

Apenas lo creás, un robot revisa el nombre de tu rama. Si está bien, no pasa nada. Si está mal, te comenta cómo arreglarlo y **cierra el PR** (no perdés nada: renombrás la rama y abrís uno nuevo). Es el único caso en que un PR del curso se cierra antes de tiempo.

Después, durante el cuatrimestre:

| Lo que ves | Qué significa | Qué hacés |
| :--- | :--- | :--- |
| 🟢 **Open**, sin marcas | Tus entregas llegaron bien | Nada |
| 💬 **Comment** | Una observación o pregunta del docente | Respondé en el mismo hilo |
| 🔴 **Changes requested** | Rechazó una entrega: hay algo que corregir | Corregí, `commit` + `push` a la **misma** rama. El push levanta la marca |
| 🟣 **Merged** | Nunca va a pasar | — |

**"Changes requested" no cierra tu PR.** Sigue abierto y sigue siendo el mismo: no abras uno nuevo ni crees otra rama. Es feedback, no una puerta cerrada.


> En los próximos ejercicios **no abrís PRs nuevos**: solo `commit` + `push` a tu rama y este mismo PR se actualiza solo. **Una rama para siempre, un PR para siempre.** El docente identifica cada entrega por el commit `ejercicioNN: ...` y el `.txt` nuevo. Detalle en el [README raíz](../../README.md).

> **El docente NO va a mergear tu PR** (tampoco al final del cuatrimestre) — y está bien que así sea: tu rama es tu espacio de trabajo y el PR es la ventana por donde se revisan tus entregas. Un PR abierto con pushes semanales es la señal de que venís bien.

> **Nota para el/la docente**: el PR del estudiante queda **abierto** todo el cuatrimestre y **nunca se mergea** (al cierre se cierra sin mergear). **NO** borres la rama `estudiante/apellido-nombre`.